In [ ]:
import polars as pl
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from datetime import datetime
import zygo_reader

# --- Configuration ---
# Path to the generated files metadata
GEN_FILES_PARQUET = R"T:\asm\E X E_MB\10-EXE_MB_Flatness_Database\.db\Generated Files\generated_files.parquet"
# Root path of the flatness database
DATABASE_PATH = r"T:\asm\E X E_MB\10-EXE_MB_Flatness_Database"
# Output CSV for analysis results
ANALYSIS_RESULTS_CSV = "analysis_results.csv"
# Output CSV for labels
LABELS_CSV = R"C:\Users\dmonacha\OneDrive - ASML\Desktop\semiconductor-anomaly-detection\data\GENERATED\labels.csv"

# --- 1. Load File Paths ---
print("Loading file paths...")
gen_files = pl.read_parquet(GEN_FILES_PARQUET)

# Filter for the specific files we need (using filter from test1.ipynb)
diff_map_files_hi = gen_files.filter(
    (pl.col("description") == "Side difference map - Zygo file") &
    (pl.col("filePath").str.contains(R".*Z.*HI.*coating"))
)

# Create full file paths
full_paths = [os.path.join(DATABASE_PATH, f) for f in diff_map_files_hi["filePath"]]
df_full_path = pd.DataFrame(full_paths, columns=["fullPath"])
print(f"Found {len(df_full_path)} files to process.")


# --- 2. Extract Labels from Paths ---
def extract_labels(path):
    path = path.replace("\\", "/")
    # Extract Process
    m1 = re.search(r'EXE_MB_Flatness_Database/([^/]+)/Difference plots/', path)
    process = m1.group(1) if m1 else None

    # Get folder under 'Difference plots'
    m2 = re.search(r'Difference plots/([^/]+)/', path)
    folder = m2.group(1) if m2 else None

    # Extract Side (look for ZA or ZE just before 'Diff(')
    m_side = re.search(r'\b(Z[EA])(?= Diff\()', path)
    side = m_side.group(1) if m_side else 'other'

    date = None
    step1 = ''
    step2 = ''

    if folder:
        parts = [p.strip() for p in folder.split(' - ')]
        date_match = re.match(r"^(\d{6})(?:\s*(\d{2})h(\d{2}))?(.*)", parts[0])
        if date_match:
            yymmdd, hour, minute, step1_extra = date_match.groups()
            step1_extra = step1_extra.strip()
            try:
                date_str = f"20{yymmdd}"
                if hour and minute:
                    date = datetime.strptime(f"{date_str} {hour}:{minute}", "%Y%m%d %H:%M")
                else:
                    date = datetime.strptime(date_str, "%Y%m%d")
            except ValueError:
                date = None

            if len(parts) == 2:
                step1 = step1_extra if step1_extra else parts[1]
                step2 = parts[1]
            elif len(parts) >= 3:
                step1_pieces = [step1_extra] if step1_extra else []
                step1_pieces += parts[1:-1]
                step1 = ' - '.join(filter(None, step1_pieces))
                step2 = parts[-1]
            else:
                step1 = step1_extra
        else:
            step1 = parts[0] if len(parts) > 0 else ''
            step2 = parts[1] if len(parts) > 1 else ''

    return pd.Series([process, side, step1.strip(), step2.strip(), date],
                     index=['Process', 'Side', 'Step 1', 'Step 2', 'Date'])

print("Extracting labels from file paths...")
df_full_path[['Process', 'Side', 'Step 1', 'Step 2', 'Date']] = df_full_path['fullPath'].apply(extract_labels)

# Save labels to CSV
labels = df_full_path[['Process', 'Step 1', 'Step 2', 'Date', 'Side']]
labels.to_csv(LABELS_CSV, index=False)
print(f"Labels saved to {LABELS_CSV}")


# --- 3. NaN Imputation Function ---
def impute_nans(height_map):
    """Impute NaN values using median fill"""
    if np.all(np.isnan(height_map)):
        return np.zeros_like(height_map)
    
    # Use median of non-NaN values
    median_val = np.nanmedian(height_map)
    if np.isnan(median_val):
        return np.zeros_like(height_map)
    
    height_map_filled = np.where(np.isnan(height_map), median_val, height_map)
    return height_map_filled


# --- 4. Process Files: Validate Data and Generate Images ---
print("\nStarting data processing and image generation...")
analysis_results = []

# Create directories for image storage
for side_folder in df_full_path['Side'].unique():
    if not os.path.exists(side_folder):
        print(f"Creating directory: {side_folder}")
        os.makedirs(side_folder)

# Single loop to process each file
for index, row in df_full_path.iterrows():
    test_file = row["fullPath"]
    side = row["Side"]
    status = 'good'
    details = {}
    height_map = None

    try:
        # Read the data file once
        zygo_data = zygo_reader.DatReader(path_or_file_like=test_file)
        height_map = zygo_data.get_topography_nm()

        # --- Data Validation ---
        if height_map is None or height_map.size == 0:
            status = 'error_empty'
        else:
            nan_percentage = np.count_nonzero(np.isnan(height_map)) / height_map.size
            details['nan_percentage'] = nan_percentage
            if nan_percentage > 0.9:
                status = 'error_mostly_nan'

            std_dev = np.nanstd(height_map)
            details['std_dev'] = std_dev
            if std_dev < 1e-6:
                status = 'error_flat'

    except Exception as e:
        status = 'error_read'
        details['read_error'] = str(e)

    # --- Image Generation (if data is good) ---
    if status == 'good' and height_map is not None:
        try:
            base_filename = os.path.basename(test_file)
            sanitized_name = re.sub(r'[\\/*?:"<>|()', '_', base_filename)
            image_filename = os.path.splitext(sanitized_name)[0] + '.png'
            output_path = os.path.join(side, image_filename)
            print(output_path)

            # --- Fix for NaN values and blank images ---
            # Handle NaN values before processing
            if np.any(np.isnan(height_map)):
                height_map = impute_nans(height_map)
                details['nan_imputed'] = True
            else:
                details['nan_imputed'] = False
            
            # Calculate robust color limits, ignoring top/bottom 1% of data as outliers
            vmin = np.nanpercentile(height_map, 1)
            vmax = np.nanpercentile(height_map, 99)
            
            # Use Matplotlib to save the array as a heatmap image. This is much faster.
            # np.flipud is used because matplotlib's origin is top-left.
            # vmin/vmax ensure the colormap is applied to the data range we care about.
            plt.imsave(output_path, height_map, cmap='rainbow_r', vmin=vmin, vmax=vmax)
            plt.close('all') # Free up memory

        except Exception as e:
            status = 'error_save'
            details['save_error'] = str(e)

    # Append analysis results for this file
    result_entry = {'filepath': test_file, 'side': side, 'status': status}
    result_entry.update(details)
    analysis_results.append(result_entry)

    if (index + 1) % 50 == 0:
        print(f"  ... processed {index + 1} / {len(df_full_path)} files")

# --- 5. Save Final Report ---
print("\nProcessing complete.")
results_df = pd.DataFrame(analysis_results)
results_df.to_csv(ANALYSIS_RESULTS_CSV, index=False)
print(f"Analysis results saved to {ANALYSIS_RESULTS_CSV}")

print("\nStatus summary:")
print(results_df['status'].value_counts())

In [ ]:
import os
import polars as pl
import plotly.express as px
import numpy as np

database_path = R"T:\asm\E X E_MB\10-EXE_MB_Flatness_Database"
gen_files = pl.read_parquet(R"T:\asm\E X E_MB\10-EXE_MB_Flatness_Database\.db\Generated Files\generated_files.parquet")
diff_map_files = gen_files.filter(pl.col("description") == "Side difference map - Zygo file")
diff_map_files_HI = diff_map_files.filter(pl.col("filePath").str.contains(R".*Z.*HI.*coating"))
diff_map_files_HI.select("filePath")


def impute_nans(height_map):
    """Impute NaN values using median fill"""
    if np.all(np.isnan(height_map)):
        return np.zeros_like(height_map)
    
    # Use median of non-NaN values
    median_val = np.nanmedian(height_map)
    if np.isnan(median_val):
        return np.zeros_like(height_map)
    
    height_map_filled = np.where(np.isnan(height_map), median_val, height_map)
    return height_map_filled


def show_zygo_images(file_paths):
    for fpath in file_paths:
        zygoData = zygo_reader.DatReader(path_or_file_like=fpath)
        height_map = zygoData.get_topography_nm()
        
        # Handle NaN values before visualization
        if np.any(np.isnan(height_map)):
            height_map = impute_nans(height_map)
            print(f"NaN values imputed for: {fpath}")
        
        fig = px.imshow(height_map, color_continuous_scale=px.colors.sequential.Rainbow_r,
                        title=fpath)
        fig.show()

# Prepare your file paths as a list:
test_files = [
    os.path.join(database_path, diff_map_files_HI.get_column("filePath")[i]) 
    for i in range(300, 310)
]
# Call the function:
import zygo_reader
show_zygo_images(test_files)